# Topographic-dominated eddies: polarity-dependent response to signed PV gradients

## Corrected hypothesis

For signed $PV=(\zeta+f)/h$, AEs are predicted to **align** with $\nabla PV$ ($\Delta\theta\to0^\circ$), whereas CEs are predicted to **oppose** it ($\Delta\theta\to180^\circ$).

The CE transition is: natural poleward tilt → departure as total signed $\nabla PV$ rotates → increasing opposition to the signed topographic gradient. The secondary AE hypothesis is that shelf proximity disrupts its otherwise aligned response. That mechanism must be tested separately rather than inferred from a broad distribution alone.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'seacofs_tilt_tools.py').exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or one of its subfolders.')
CASE_ROOT = ANALYSIS_ROOT / 'case_studies'
for path in (ANALYSIS_ROOT, CASE_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import seacofs_tilt_tools as tilt
from case_study_tools import PVAlignmentConfig, add_pv_alignment_diagnostics

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 60)

## 1. Load data and define the topographic-dominated population

In [ ]:
DOMINANCE_FACTOR = 2.0
MAX_DEPTH_m = 3000.0
MIN_TILT_km = 5.0
ALIGNMENT_DEG = 30.0
SHELF_LON = 154.75

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df, _ = tilt.load_tilt_tables(paths)
df = tilt.add_region_labels(df, grid)
df = tilt.add_pv_gradient_terms(df, grid, core_mean=True)

config = PVAlignmentConfig(
    dominance_factor=DOMINANCE_FACTOR, max_topo_depth_m=MAX_DEPTH_m,
    min_tilt_distance_km=MIN_TILT_km,
)
d = add_pv_alignment_diagnostics(df, config)
assert d.beta.dropna().median() > 0, 'Expected positive planetary beta after gradient sign correction.'
topographic = d[d.shallow_ocean_topographic & d.direction_valid].copy()
topographic = topographic[np.isfinite(topographic.PV_grad_mag) & (topographic.PV_grad_mag > 0)]
topographic['log_PV_grad_mag'] = np.log(topographic.PV_grad_mag)
topographic['raw_signed_PV_mismatch'] = topographic.dtheta_PV_grad
topographic['preference_error'] = np.where(topographic.Cyc.eq('AE'),
    topographic.raw_signed_PV_mismatch, 180.0-topographic.raw_signed_PV_mismatch)
topographic['predicted_30'] = topographic.preference_error <= ALIGNMENT_DEG
topographic['response_cosine'] = np.cos(np.deg2rad(topographic.raw_signed_PV_mismatch))
topographic['shelf_class'] = np.where(topographic.lon < SHELF_LON, 'On-shelf', 'Off-shelf')

display(topographic.groupby(['Cyc', 'shelf_class']).agg(
    observations=('Eddy', 'size'), eddies=('Eddy', 'nunique'),
    median_PV_grad=('PV_grad_mag', 'median'), median_raw_mismatch_deg=('raw_signed_PV_mismatch','median'),
    median_preference_error_deg=('preference_error','median'),
    predicted_30_fraction=('predicted_30','mean')
).round(3))

In [ ]:
topographic.head()

## 2. First verify the claimed AE–CE gradient contrast

The six-fold contrast is treated as an empirical claim to estimate, not as an input assumption. Medians are calculated per eddy before comparing polarities.

In [ ]:
eddy_pv = (topographic.groupby(['Cyc', 'Eddy'], as_index=False)
           .agg(median_PV_grad=('PV_grad_mag', 'median'), observations=('Day', 'size')))
pv_summary = eddy_pv.groupby('Cyc').median(numeric_only=True)
display(pv_summary)
ce_ae_ratio = (pv_summary.loc['CE', 'median_PV_grad'] / pv_summary.loc['AE', 'median_PV_grad'])
print(f'Ratio of CE to AE median eddy-level |∇PV|: {ce_ae_ratio:.2f}')

fig, ax = plt.subplots(figsize=(5, 3.5), constrained_layout=True)
for cyc, color in [('AE', 'tab:red'), ('CE', 'tab:blue')]:
    x = np.log(eddy_pv.loc[eddy_pv.Cyc == cyc, 'median_PV_grad'])
    ax.hist(x, bins='fd', density=True, histtype='step', lw=2, color=color, label=cyc)
ax.set(xlabel=r'eddy median $\log |\nabla \mathrm{PV}|$', ylabel='Density')
ax.legend(frameon=False);

## 3. Map and alignment distributions

A common normalization is used for the AE and CE maps. Separate colour families identify polarity, but equal normalized intensity represents equal `log|∇PV|`.

In [ ]:
logpv = topographic.log_PV_grad_mag
norm = Normalize(*logpv.quantile([.01, .99]))
fig, axs = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)
scats = []
for ax, cyc, cmap in zip(axs, ['AE', 'CE'], ['Reds', 'Blues']):
    part = topographic[topographic.Cyc == cyc]
    ax.contourf(grid.X_grid, grid.Y_grid, np.where(grid.mask_rho, grid.h / 1e3, np.nan), cmap='Greys_r')
    sc = ax.scatter(part.xc, part.yc, c=part.log_PV_grad_mag, cmap=cmap, norm=norm, s=4, alpha=.55, edgecolors='none')
    scats.append(sc)
    ax.contour(grid.X_grid, grid.Y_grid, grid.h, levels=[MAX_DEPTH_m], colors='gold')
    tilt.lat_lon_contours(ax, grid)
    ax.set(title=cyc, xlabel='x (km)', aspect='equal')
    fig.colorbar(sc, ax=ax, location='top', label=r'$\log |\nabla \mathrm{PV}|$', fraction=.05, pad=.02)
axs[0].set_ylabel('y (km)');

In [ ]:
bins = np.arange(0, 181, 10)
fig, axs = plt.subplots(2, 2, figsize=(8, 6), sharex=True, sharey=True, constrained_layout=True)
for i, cyc in enumerate(['AE', 'CE']):
    # On-shelf tercile edges are the reference for both columns in this polarity.
    ref = topographic[(topographic.Cyc == cyc) & (topographic.shelf_class == 'On-shelf')]
    # q1, q2 = ref.PV_grad_mag.quantile([1/3, 2/3])

    q2 = 1.995353994964308e-12
    q1 = 1.916967797095701e-13
    
    edges = [-np.inf, q1, q2, np.inf]
    cmap = plt.get_cmap('Reds' if cyc == 'AE' else 'Blues')
    for j, shelf in enumerate(['On-shelf', 'Off-shelf']):
        ax = axs[i, j]
        part = topographic[(topographic.Cyc == cyc) & (topographic.shelf_class == shelf)].copy()
        print(part[part.preference_error>90].PV_grad_mag.mean())
        part['PV_bin'] = pd.cut(part.PV_grad_mag, edges, labels=['Low', 'Medium', 'High'])
        groups, colors = [], []
        for label in ['Low', 'Medium', 'High']:
            group = part[part.PV_bin == label]
            groups.append(group.dtheta_PV_grad)
            colors.append(cmap(norm(group.log_PV_grad_mag.median())))
        weights = [np.ones(len(x)) * 100 / len(part) for x in groups]
        ax.hist(groups, bins=bins, weights=weights, stacked=True, color=colors)
        ax.axvspan(0, ALIGNMENT_DEG, color='0.5', alpha=.12)
        ax.axvline(90, color='0.4', lw=.8)
        ax.set(title=f'{cyc} {shelf}', xlim=(0, 180), xticks=[0, 45, 90, 135, 180])
        if i == 1: ax.set_xlabel(r'$|\theta_{tilt}-\theta_{\nabla PV}|$ (°)')
        if j == 0: ax.set_ylabel('Observations (%)')

In [ ]:
bins = np.arange(0, 181, 10)
fig, axs = plt.subplots(2, 2, figsize=(8,6), sharex=True, sharey=True, constrained_layout=True)

for i, cyc in enumerate(['AE','CE']):
    cmap = plt.get_cmap('Reds' if cyc=='AE' else 'Blues')
    for j, shelf in enumerate(['On-shelf','Off-shelf']):
        ax = axs[i,j]
        part = topographic[(topographic.Cyc==cyc) & (topographic.shelf_class==shelf)].copy()
        aligned = part.preference_error < 90
        ts = part.PV_grad_mag.quantile(np.linspace(.05,.95,200)).values
        t = max(ts, key=lambda x: (part.PV_grad_mag>x)[aligned].mean()-(part.PV_grad_mag>x)[~aligned].mean())
        part['PV_bin'] = part.PV_grad_mag > t
        groups = [part.loc[~part.PV_bin,'dtheta_PV_grad'], part.loc[part.PV_bin,'dtheta_PV_grad']]
        ax.hist(groups, bins=bins, weights=[np.ones(len(x))*100/len(part) for x in groups],
                stacked=True, color=[cmap(.3),cmap(.8)])
        ax.axvspan(0, ALIGNMENT_DEG, color='0.5', alpha=.12); ax.axvline(90,color='0.4',lw=.8)
        ax.set(title=f'{cyc} {shelf} ({t:.1e})', xlim=(0,180), xticks=[0,45,90,135,180])
        if i==1: ax.set_xlabel(r'$|\theta_{tilt}-\theta_{\nabla PV}|$ (°)')
        if j==0: ax.set_ylabel('Observations (%)')

In [ ]:
bins = np.arange(0,181,10)
fig, axs = plt.subplots(2,2,figsize=(8,6),sharex=True,sharey=True,constrained_layout=True)

for i,cyc in enumerate(['AE','CE']):
    ref = topographic[topographic.Cyc==cyc]
    a = ref.preference_error<90
    ts = ref.PV_grad_mag.quantile(np.linspace(.05,.95,200))
    t = max(ts,key=lambda x:(ref.PV_grad_mag>x)[a].mean()-(ref.PV_grad_mag>x)[~a].mean())
    cmap = plt.get_cmap('Reds' if cyc=='AE' else 'Blues')

    for j,shelf in enumerate(['On-shelf','Off-shelf']):
        ax = axs[i,j]
        part = ref[ref.shelf_class==shelf].copy()
        part['high'] = part.PV_grad_mag>t
        groups = [part.loc[~part.high,'dtheta_PV_grad'],part.loc[part.high,'dtheta_PV_grad']]
        ax.hist(groups,bins=bins,weights=[np.ones(len(x))*100/len(part) for x in groups],
                stacked=True,color=[cmap(.3),cmap(.8)],label=['Low','High'])
        ax.axvspan(0,ALIGNMENT_DEG,color='0.5',alpha=.12); ax.axvline(90,color='0.4',lw=.8)
        ax.set(title=f'{cyc} {shelf}',xlim=(0,180),xticks=[0,45,90,135,180])
        if i==1: ax.set_xlabel(r'$|\theta_{tilt}-\theta_{\nabla PV}|$ (°)')
        if j==0: ax.set_ylabel('Observations (%)')

axs[0,1].legend(frameon=False)



## 4. Primary test: conditional polarity response versus PV-gradient magnitude

Equal-population bins show both the median angular mismatch and the probability of the polarity-predicted response. A forcing threshold would appear as a relatively abrupt decline in angle and rise in alignment probability, especially for AEs. A smooth trend would instead support progressive constraint without a sharp threshold.

In [ ]:
def binned_alignment(part, n_bins=12):
    part = part.copy()
    part['PV_bin'] = pd.qcut(part.log_PV_grad_mag, n_bins, duplicates='drop')
    return (part.groupby('PV_bin', observed=True)
            .agg(logPV=('log_PV_grad_mag', 'median'), angle=('preference_error','median'),
                 q25=('dtheta_PV_grad', lambda x: x.quantile(.25)),
                 q75=('dtheta_PV_grad', lambda x: x.quantile(.75)),
                 p_aligned=('predicted_30','mean'), n=('Eddy', 'size'), eddies=('Eddy', 'nunique'))
            .reset_index(drop=True))

fig, axs = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for cyc, color in [('AE', 'tab:red'), ('CE', 'tab:blue')]:
    stats = binned_alignment(topographic[topographic.Cyc == cyc])
    axs[0].plot(stats.logPV, stats.angle, '-o', color=color, label=cyc)
    axs[0].fill_between(stats.logPV, stats.q25, stats.q75, color=color, alpha=.15)
    axs[1].plot(stats.logPV, 100 * stats.p_aligned, '-o', color=color, label=cyc)
axs[0].axhline(90, color='0.5', ls=':', lw=.8)
axs[0].axhline(ALIGNMENT_DEG, color='0.5', ls='--', lw=.8)
axs[0].set(xlabel=r'$\log |\nabla \mathrm{PV}|$', ylabel='Median error from predicted direction (°)', ylim=(0, 180))
axs[1].set(xlabel=r'$\log |\nabla \mathrm{PV}|$', ylabel=rf'$P(\Delta\theta \leq {ALIGNMENT_DEG:.0f}^\circ)$ (%)', ylim=(0, 100))
for ax in axs: ax.legend(frameon=False)

## 5. Does tilt magnitude change the result?

The 5 km cutoff removes the least meaningful bearings. This sensitivity plot repeats the relationship for progressively larger tilt displacements; a physical alignment signal should not be created solely by the cutoff.

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(12, 6), sharex='col', sharey=True, constrained_layout=True)
for j, min_tilt in enumerate([5, 10, 20]):
    for i, (cyc, color) in enumerate([('AE', 'tab:red'), ('CE', 'tab:blue')]):
        part = topographic[(topographic.Cyc == cyc) & (topographic.TiltDis >= min_tilt)]
        stats = binned_alignment(part, n_bins=10)
        axs[i, j].plot(stats.logPV, 100 * stats.p_aligned, '-o', color=color)
        axs[i, j].set(title=f'{cyc}: TiltDis ≥ {min_tilt} km', ylim=(0, 100))
        if i == 1: axs[i, j].set_xlabel(r'$\log |\nabla \mathrm{PV}|$')
        if j == 0: axs[i, j].set_ylabel('Polarity-predicted observations (%)')

## 6. Eddy-level trend estimates

Estimate one slope per eddy where enough gradient variation exists. The distribution of slopes asks whether the relationship recurs across eddies rather than being driven by a few long tracks. Negative preference-error slopes and positive predicted-response slopes support the hypothesis.

In [ ]:
slope_rows = []
for (cyc, eddy), part in topographic.groupby(['Cyc', 'Eddy']):
    if len(part) < 10 or part.log_PV_grad_mag.nunique() < 4:
        continue
    angle_slope = np.polyfit(part.log_PV_grad_mag, part.preference_error, 1)[0]
    aligned_slope = np.polyfit(part.log_PV_grad_mag, part.predicted_30.astype(float), 1)[0]
    slope_rows.append({'Cyc': cyc, 'Eddy': eddy, 'n': len(part),
                       'angle_slope_deg_per_logPV': angle_slope,
                       'response_slope_per_logPV': aligned_slope})
slopes = pd.DataFrame(slope_rows)
display(slopes.groupby('Cyc').agg(
    eddies=('Eddy', 'size'), median_angle_slope=('angle_slope_deg_per_logPV', 'median'),
    fraction_negative_angle=('angle_slope_deg_per_logPV', lambda x: (x < 0).mean()),
    median_alignment_slope=('response_slope_per_logPV', 'median'),
    fraction_positive_alignment=('response_slope_per_logPV', lambda x: (x > 0).mean())
).round(3))

## 7. Interpretation and next statistical model

A convincing result requires agreement among: (1) binned angle distributions, (2) alignment probability, (3) tilt-distance sensitivity, and (4) within-eddy slopes. A formal follow-up should use a mixed-effects or clustered model with eddy as the grouping unit, allow nonlinear dependence on `log|∇PV|`, and test a polarity-by-gradient interaction. A segmented model should only be used to claim a threshold if it materially improves out-of-sample fit over a smooth monotonic alternative.